In [1]:
# Import essential libraries for geospatial analysis and visualization
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
from sentence_transformers import SentenceTransformer, util
from sklearn.cluster import DBSCAN
from scipy.spatial import cKDTree
import torch

'''
--------------------------------------------------------------
The following libraries are used for geospatial analysis and machine learning:
- geopandas: For handling geospatial data (version 0.14.1)
- pandas: For data manipulation and analysis (version 2.1.1)
- numpy: For numerical operations (version 1.26.0)
- shapely: For geometric operations (version 2.0.1)
- sentence_transformers: For text embeddings (version 2.2.2)
- scikit-learn: For clustering algorithms (version 1.3.0)
- scipy: For spatial data processing (version 1.11.3)
- torch: For tensor operations (version 2.1.0)

Citations:
GeoPandas developers (2023) GeoPandas: Python tools for geographic data (Version 0.14.1) [Computer program].
Available at: https://geopandas.org (Accessed: 2 May 2025).

Pedregosa, F. et al. (2011) 'Scikit-learn: Machine Learning in Python', Journal of Machine Learning Research,
12, pp. 2825-2830.
--------------------------------------------------------------
'''

from google.colab import drive
drive.mount('/content/drive')

# Fixed user location for the center of Birmingham
USER_LON, USER_LAT = -1.900, 52.480
USER_POINT = Point(USER_LON, USER_LAT)

# I'm loading the shapefile containing Points of Interest (POIs) in Birmingham
# and transforming the coordinate system to British National Grid (EPSG:27700)
print("Loading data...")
shapefile_path = "/content/drive/MyDrive/GeoAI/shapefiles/Burm_poi.shp"
gdf_original = gpd.read_file(shapefile_path).to_crs("EPSG:27700")
gdf_original["x"], gdf_original["y"] = gdf_original.geometry.x, gdf_original.geometry.y
print(f"Loaded {len(gdf_original):,} POIs")

# Checking the first few records to understand the structure of the data
print("Sample data:")
print(gdf_original.head()[["name", "fclass", "geometry"]])

Mounted at /content/drive
Loading data...
Loaded 20,922 POIs
Sample data:
   name               fclass                       geometry
0  None  camera_surveillance  POINT (406053.833 292523.031)
1  None             post_box   POINT (412510.12 294742.914)
2  None             post_box  POINT (412075.789 295193.211)
3  None             post_box  POINT (411740.959 295186.853)
4  None            telephone  POINT (411750.036 295176.407)


In [2]:
# I'm loading the Sentence-BERT model which will help me calculate semantic
# similarity between user queries and POI descriptions
print("Loading SBERT model...")
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

'''
--------------------------------------------------------------
The Sentence-BERT model is used for creating text embeddings:
Reimers, N. and Gurevych, I. (2019) 'Sentence-BERT: Sentence Embeddings using
Siamese BERT-Networks', Proceedings of the 2019 Conference on Empirical Methods in
Natural Language Processing and the 9th International Joint Conference on Natural
Language Processing (EMNLP-IJCNLP). Available at: https://arxiv.org/abs/1908.10084
(Accessed: 2 May 2025).
--------------------------------------------------------------
'''

# Creating text embeddings by combining the name and fclass fields
# These embeddings will be used to measure semantic similarity with user queries
print("Creating text embeddings...")
poi_texts = (gdf_original["name"].fillna("") + " " + gdf_original["fclass"].fillna("")).str.strip().tolist()
poi_emb = sbert_model.encode(poi_texts, convert_to_numpy=True, normalize_embeddings=True)
gdf_original["embedding"] = list(poi_emb)
print("Embeddings created and cached")

Loading SBERT model...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Creating text embeddings...
Embeddings created and cached


In [3]:
# I'm defining a function to compute topicality scores based on semantic similarity
def compute_topicality(gdf, q_text):
    """Calculate topicality scores using SBERT"""
    gdf = gdf.copy()
    # Encode the query text into an embedding vector
    q_emb = sbert_model.encode([q_text], convert_to_numpy=True, normalize_embeddings=True)[0]

    # Calculate cosine similarity between query and POI embeddings
    # I'm using a dot product since vectors are normalized (equivalent to cosine similarity)
    sims = np.vstack(gdf["embedding"].values) @ q_emb

    # Normalize scores to range [0,1] for fair comparison with other scores
    if sims.max() > sims.min():
        gdf["S_topicality"] = (sims - sims.min()) / (sims.max() - sims.min())
    else:
        gdf["S_topicality"] = 0.0
    return gdf

# I'm defining a function to calculate the Haversine distance between two points on Earth
def haversine_m(p, q):
    """Calculate Haversine distance between two points in meters"""
    R = 6_371_008.8  # Earth radius in meters
    lat1, lon1 = np.radians(p.y), np.radians(p.x)
    lat2, lon2 = np.radians(q.y), np.radians(q.x)
    dlat, dlon = lat2 - lat1, lon2 - lon1

    # Haversine formula implementation
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

'''
--------------------------------------------------------------
The Haversine formula implementation is modified from:
GeoPy contributors (2023) 'GeoPy - A Python library for geocoding services' (Version 2.4.0)
Available at: https://github.com/geopy/geopy
Downloaded: 15 March 2025
--------------------------------------------------------------
'''

# I'm defining a function to compute spatial proximity scores using exponential decay
def compute_spatial(gdf, user_point, decay_km=2):
    """Calculate spatial proximity scores using exponential decay"""
    gdf = gdf.copy()

    # Convert to WGS84 for accurate distance calculation
    d = gdf.to_crs("EPSG:4326").geometry.apply(lambda g: haversine_m(g, user_point))

    # Apply exponential decay function to distance
    # POIs closer to the user location get higher scores
    gdf["S_spatial"] = np.exp(-(d / (decay_km * 1000)))
    return gdf

In [4]:
# I'm defining a function to identify clusters of similar POIs using DBSCAN algorithm
def compute_cluster(gdf, poi_type, eps_m=40, min_samples=3):
    """Calculate clustering scores using DBSCAN"""
    gdf = gdf.copy()

    # Initialize the clustering score column
    if "S_cluster" not in gdf.columns:
        gdf["S_cluster"] = 0.0

    # If no POI type specified, return with zeros
    if not poi_type:
        print("Clustering: No category specified")
        return gdf

    # Extract POIs of the specified type for clustering
    sub = gdf[gdf["fclass"] == poi_type].copy()

    # If too few POIs match exactly, try a broader match
    if len(sub) < min_samples:
        print(f"Clustering: Only {len(sub)} POIs of type '{poi_type}' - trying broader match")
        sub = gdf[gdf["fclass"].str.contains(poi_type, case=False, na=False)].copy()
        print(f"  Found {len(sub)} POIs with broader match")

    # Only proceed if we have enough POIs for meaningful clustering
    if len(sub) >= min_samples:
        # Apply DBSCAN algorithm to identify clusters
        db = DBSCAN(eps=eps_m, min_samples=min_samples)
        coords = sub[["x", "y"]].values
        sub["cluster"] = db.fit_predict(coords)

        '''
        --------------------------------------------------------------
        The DBSCAN clustering algorithm is implemented using:
        Pedregosa, F. et al. (2011) 'Scikit-learn: Machine Learning in Python',
        Journal of Machine Learning Research, 12, pp. 2825-2830.

        The algorithm is described in:
        Ester, M., Kriegel, H.P., Sander, J. and Xu, X. (1996) 'A density-based
        algorithm for discovering clusters in large spatial databases with noise',
        Proceedings of the Second International Conference on Knowledge Discovery
        and Data Mining (KDD-96), pp. 226-231.
        --------------------------------------------------------------
        '''

        # Process clusters and assign scores based on cluster membership
        valid_clusters = sub[sub["cluster"] >= 0]["cluster"]
        if len(valid_clusters) > 0:
            sizes = valid_clusters.value_counts()
            print(f"Clustering: Found {len(sizes)} clusters with max size {sizes.max()}")

            # Normalize cluster sizes - POIs in larger clusters get higher scores
            if sizes.max() > 0:
                sub["S_cluster"] = sub["cluster"].map(
                    lambda c: sizes.get(c, 0) / sizes.max() if c >= 0 else 0
                )
                # Transfer scores back to main dataframe
                gdf.loc[sub.index, "S_cluster"] = sub["S_cluster"]
            else:
                print("Clustering: No valid clusters found")
        else:
            print("Clustering: All points classified as noise")
    else:
        print(f"Clustering: Too few POIs ({len(sub)}) for clustering")

    return gdf

In [5]:
# I'm defining a function to identify POIs that are located near other POIs of interest
def compute_coloc(gdf, a, b, radius_m=120):
    """Calculate co-location scores based on spatial proximity"""
    gdf = gdf.copy()

    # Initialize the colocation score column
    if "S_coloc" not in gdf.columns:
        gdf["S_coloc"] = 0.0

    # Skip if either category is missing
    if not a or not b:
        print("Co-location: Missing category parameter")
        return gdf

    # Get POIs of each type
    A = gdf[gdf["fclass"] == a].copy()
    B = gdf[gdf["fclass"] == b].copy()

    # If exact match doesn't yield results, try a broader match for buddy category
    if B.empty:
        print(f"Co-location: No POIs of type '{b}' found in dataset")
        # Try a more flexible match using string contains
        B = gdf[gdf["fclass"].str.contains(b, case=False, na=False)].copy()
        print(f"  Trying partial match: found {len(B)} POIs")

    # Skip if either set is empty
    if A.empty or B.empty:
        print(f"Co-location: Missing POIs (type A: {len(A)}, type B: {len(B)})")
        return gdf

    # Use KDTree for efficient spatial proximity queries
    tree = cKDTree(B[["x", "y"]].values)

    '''
    --------------------------------------------------------------
    The KDTree implementation for spatial queries is from:
    SciPy developers (2023) 'SciPy: Scientific Library for Python' (Version 1.11.3)
    Available at: https://scipy.org/
    Downloaded: 15 March 2025
    --------------------------------------------------------------
    '''

    # Count how many POIs of type B are within radius of each POI of type A
    counts = tree.query_ball_point(A[["x", "y"]].values, r=radius_m, return_length=True)

    # Normalize counts if there's variation
    if max(counts) > min(counts):
        A["S_coloc"] = (counts - min(counts)) / (max(counts) - min(counts))
        gdf.loc[A.index, "S_coloc"] = A["S_coloc"]
        print(f"Co-location: min={min(counts)}, max={max(counts)}")
    else:
        print(f"Co-location: all counts same value ({min(counts) if len(counts) > 0 else 0})")

    return gdf

In [6]:
# I'm defining a function to guess OSM category from user queries
def guess_category_from_query(q):
    """Guess OSM category from user query"""
    q = (q or "").lower()

    # Map common terms to OSM categories
    if "restaurant" in q: return "restaurant"
    if "barber" in q or "hair" in q: return "hairdresser"
    if "cafe" in q or "coffee" in q: return "cafe"
    if "school" in q: return "school"
    if "bank" in q: return "bank"
    if "shop" in q or "store" in q: return "supermarket"
    if "pub" in q or "bar" in q: return "pub"
    if "hotel" in q: return "hotel"
    if "park" in q: return "park"
    if "beauty" in q: return "beauty"

    # Fallback to first word if no pattern match
    return q.split()[0] if q else None

# I'm defining the main pipeline function that integrates all scoring components
def rank_pois_pipeline(
    gdf,
    query_text,
    query_cat=None,
    buddy_cat=None,
    user_lonlat=(USER_LON, USER_LAT),
    weights=(0.6, 0.25, 0.1, 0.05)
):
    """Main pipeline to calculate Geographic Relevance scores"""
    # Start with fresh copy to avoid modifying original data
    gdf = gdf.copy()

    # Auto-detect category if not provided
    if not query_cat:
        query_cat = guess_category_from_query(query_text)

    print(f"Query: '{query_text}', Category: '{query_cat}', Buddy: '{buddy_cat}'")
    print(f"User location: {user_lonlat}")

    # 1. Compute topicality scores based on semantic similarity
    gdf = compute_topicality(gdf, query_text)

    # 2. Compute spatial proximity scores based on distance
    user_point = Point(user_lonlat[0], user_lonlat[1])  # lon, lat order
    gdf = compute_spatial(gdf, user_point)

    # 3. Compute clustering scores based on density of similar POIs
    gdf = compute_cluster(gdf, query_cat)

    # 4. Initialize colocation score column
    if "S_coloc" not in gdf.columns:
        gdf["S_coloc"] = 0.0

    # 5. Compute colocation scores if buddy category provided
    if buddy_cat:
        gdf = compute_coloc(gdf, a=query_cat, b=buddy_cat)

    # 6. Adjust weights if no buddy category is provided
    if not buddy_cat:
        α, β, γ, δ = (0.6, 0.3, 0.1, 0.0)
    else:
        α, β, γ, δ = weights

    # 7. Calculate final relevance score as weighted sum of components
    gdf["R"] = (
        α * gdf["S_topicality"].fillna(0) +
        β * gdf["S_spatial"].fillna(0) +
        γ * gdf["S_cluster"].fillna(0) +
        δ * gdf["S_coloc"].fillna(0)
    )

    '''
    --------------------------------------------------------------
    The geographic relevance scoring approach is adapted from:
    De Sabbata, S. and Reichenbacher, T. (2012) 'Criteria of geographic
    relevance: an experimental study', International Journal of Geographical
    Information Science, 26(8), pp. 1495-1520.
    DOI: 10.1080/13658816.2011.639303
    --------------------------------------------------------------
    '''

    # Output score statistics for debugging
    print("\nScore statistics:")
    print(f"Topicality: min={gdf['S_topicality'].min():.3f}, max={gdf['S_topicality'].max():.3f}")
    print(f"Spatial: min={gdf['S_spatial'].min():.3f}, max={gdf['S_spatial'].max():.3f}")
    print(f"Cluster: min={gdf['S_cluster'].min():.3f}, max={gdf['S_cluster'].max():.3f}")
    print(f"Colocation: min={gdf['S_coloc'].min():.3f}, max={gdf['S_coloc'].max():.3f}")
    print(f"Final R: min={gdf['R'].min():.3f}, max={gdf['R'].max():.3f}")

    # Return POIs sorted by relevance score
    return gdf.sort_values("R", ascending=False)

In [7]:
# I'm testing the pipeline with a barber shop query to see the results
results = rank_pois_pipeline(
    gdf=gdf_original,
    query_text="barber shop",
    query_cat="hairdresser",
    buddy_cat="beauty",
    user_lonlat=(USER_LON, USER_LAT),
    weights=(0.6, 0.25, 0.1, 0.05)
)

# Display the top results to examine which POIs are ranked highest
print("\nTop 15 results:")
print(results[["name", "fclass", "S_topicality", "S_spatial", "S_cluster", "S_coloc", "R"]].head(15).round(3))

# I'm also testing the pipeline without a buddy category to compare results
results_no_buddy = rank_pois_pipeline(
    gdf=gdf_original,
    query_text="restaurant",
    query_cat="restaurant",
    user_lonlat=(USER_LON, USER_LAT)
)

# Display top restaurant results without colocation score
print("\nTop 15 results (no buddy):")
print(results_no_buddy[["name", "fclass", "S_topicality", "S_spatial", "S_cluster", "R"]].head(15).round(3))

Query: 'barber shop', Category: 'hairdresser', Buddy: 'beauty'
User location: (-1.9, 52.48)
Clustering: Found 42 clusters with max size 8
Co-location: No POIs of type 'beauty' found in dataset
  Trying partial match: found 252 POIs
Co-location: min=0, max=5

Score statistics:
Topicality: min=0.000, max=1.000
Spatial: min=0.000, max=0.993
Cluster: min=0.000, max=1.000
Colocation: min=0.000, max=1.000
Final R: min=0.009, max=0.780

Top 15 results:
                        name       fclass  S_topicality  S_spatial  S_cluster  \
16225       The Barber House  hairdresser         0.900      0.963      0.000   
7136        Everyman Barbers  hairdresser         0.739      0.970      0.000   
7128             Urban Roots  hairdresser         0.745      0.938      0.000   
9256          GT Barber Shop  hairdresser         0.921      0.489      0.000   
18874          Kings Barbers  hairdresser         0.802      0.786      0.000   
20228  Waterloo Road Barbers  hairdresser         0.851      0.1

In [8]:
# I'm installing the required packages for the web interface
# Dash provides a framework for building web applications
# Dash Bootstrap Components adds styling templates
# Dash Leaflet provides interactive mapping capabilities
!pip install dash dash-bootstrap-components dash-leaflet

# Import visualization libraries
import dash
from dash import Dash, html, dcc, Input, Output, State, dash_table
import dash_bootstrap_components as dbc
import dash_leaflet as dl
import contextlib
import socket

'''
--------------------------------------------------------------
The following visualization libraries are used for creating the web interface:
- Dash: For building interactive web applications (version 3.0.4)
- Dash Bootstrap Components: For responsive layout components (version 2.0.2)
- Dash Leaflet: For interactive maps (version 1.0.15)

Citations:
Plotly Technologies Inc. (2023) 'Dash: A web application framework for Python' (Version 3.0.4)
Available at: https://dash.plotly.com/ (Accessed: 2 May 2025).

Bootstrap contributors (2023) 'Dash Bootstrap Components' (Version 2.0.2)
Available at: https://dash-bootstrap-components.opensource.faculty.ai/ (Accessed: 2 May 2025).
--------------------------------------------------------------
'''

# Constants - already defined upstream
USER_LON, USER_LAT = -1.900, 52.480

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.9/202.9 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.5/285.5 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.0/228.0 kB 15.2 MB/s eta 0:00:00
  Attempting uninstall: Werkzeug
    Found existing installation: Werkzeug 3.1.3
    Uninstalling Werkzeug-3.1.3:
      Successfully uninstalled Werkzeug-3.1.3
  Attempting uninstall: Flask
    Found existing installation: Flask 3.1.0
    Uninstalling Flask-3.1.0:
      Successfully uninstalled Flask-3.1.0


In [9]:
# I'm defining a function to create interactive map markers with popups
def make_markers(ranked, top_n=50):
    """Convert ranked POIs to leaflet markers with color coding"""
    # Convert to WGS84 for Leaflet and take top N results
    ranked = ranked.head(top_n).to_crs("EPSG:4326")
    min_r, max_r = ranked["R"].min(), ranked["R"].max()
    markers = []

    for _, row in ranked.iterrows():
        # Calculate normalized score for color gradient
        t = (row["R"] - min_r) / (max_r - min_r) if max_r > min_r else 0.5

        # Apply color gradient: red (low) to green (high)
        # I'm using a temperature-like color scheme for intuitive visualization
        if t < 0.5:
            r, g = 255, int(255 * t * 2)
        else:
            r, g = int(255 * (1 - (t - 0.5) * 2)), 255
        color = f"#{r:02x}{g:02x}00"

        # Create detailed popup with score breakdown
        popup_content = html.Div([
            html.H6(row["name"] or f"Unnamed {row['fclass'].title()}", className="mb-2"),
            html.P(f"Category: {row['fclass'].replace('_', ' ').title()}", className="mb-2 text-muted"),
            html.Hr(className="my-2"),
            html.P("Score Breakdown:", className="mb-1 fw-bold"),
            html.Div([
                html.Div([
                    "Topicality: ",
                    html.Span(f"{row['S_topicality']:.2f}", className="fw-bold")
                ], className="mb-1"),
                html.Div([
                    "Spatial: ",
                    html.Span(f"{row['S_spatial']:.2f}", className="fw-bold")
                ], className="mb-1"),
                html.Div([
                    "Cluster: ",
                    html.Span(f"{row['S_cluster']:.2f}", className="fw-bold")
                ], className="mb-1"),
                html.Div([
                    "Colocation: ",
                    html.Span(f"{row['S_coloc']:.2f}", className="fw-bold")
                ], className="mb-1"),
                html.Div([
                    "Final Score: ",
                    html.Span(f"{row['R']:.3f}", className="fw-bold")
                ], className="mt-2 text-primary")
            ])
        ], className="p-2", style={"minWidth": "200px"})

        # Create the marker with popup
        markers.append(
            dl.CircleMarker(
                center=[row.geometry.y, row.geometry.x],
                radius=8,
                color=color,
                fillColor=color,
                fillOpacity=0.7,
                weight=2,
                children=[dl.Popup(popup_content)]
            )
        )

    return markers

In [10]:
# I'm creating the user interface layout for the web application
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = html.Div([
    # Header with title and description
    html.Div([
        html.H1("Geographic Relevance POI Finder", className="display-4 text-center mb-3"),
        html.P([
            "This app uses a multi-criteria approach to find geographically relevant Points of Interest (POIs) in Birmingham.",
            html.Br(),
            "Developed by Student 209036154 - University of Leicester, GeoAI Module (GY7708)"
        ], className="lead text-center mb-3"),
    ], className="jumbotron p-4 bg-light rounded-3 mb-4"),

    # Main content container
    dbc.Container([
        dbc.Row([
            # Left column (inputs)
            dbc.Col([
                html.Div([
                    html.H4("Search Options", className="mb-3"),

                    # First input box with explanation
                    html.Div([
                        html.Label("What are you looking for?", className="form-label"),
                        dbc.Input(
                            id="search-input",
                            value="barber shop",
                            placeholder="e.g. restaurant, cafe, barber shop",
                            className="mb-2"
                        ),
                        html.Small([
                            "Main search query - determines topicality, proximity, and clustering scores.",
                            html.Br(),
                            "Examples: restaurant, cafe, barber shop, pub, hotel"
                        ], className="text-muted mb-3")
                    ], className="mb-3"),

                    # Second input box with explanation
                    html.Div([
                        html.Label("Nearby Category (optional)", className="form-label"),
                        dbc.Input(
                            id="buddy-input",
                            value="beauty",
                            placeholder="e.g. pub, beauty, cafe",
                            className="mb-2"
                        ),
                        html.Small([
                            "Finds POIs that are often located near related facilities.",
                            html.Br(),
                            "Examples: 'beauty' for barber shops, 'pub' for restaurants",
                            html.Br(),
                            "Leave empty to ignore colocation factor."
                        ], className="text-muted mb-3")
                    ], className="mb-3"),

                    # Weight sliders for adjusting criterion importance
                    html.Div([
                        html.H5("Ranking Priorities", className="mb-2"),
                        html.P("Adjust the importance of each criterion:", className="text-muted small"),

                        html.Label("Topicality (text matching)", className="form-label mt-2 mb-0"),
                        dcc.Slider(id="weight-topicality", min=0.1, max=0.8, step=0.1, value=0.6,
                                  marks={0.1: '0.1', 0.4: '0.4', 0.8: '0.8'},
                                  className="mb-2"),

                        html.Label("Spatial Proximity", className="form-label mt-2 mb-0"),
                        dcc.Slider(id="weight-spatial", min=0.1, max=0.8, step=0.1, value=0.25,
                                  marks={0.1: '0.1', 0.4: '0.4', 0.8: '0.8'},
                                  className="mb-2"),

                        html.Label("Clustering", className="form-label mt-2 mb-0"),
                        dcc.Slider(id="weight-cluster", min=0.0, max=0.5, step=0.05, value=0.1,
                                  marks={0.0: '0', 0.25: '0.25', 0.5: '0.5'},
                                  className="mb-2"),

                        html.Label("Colocation", className="form-label mt-2 mb-0"),
                        dcc.Slider(id="weight-coloc", min=0.0, max=0.5, step=0.05, value=0.05,
                                  marks={0.0: '0', 0.25: '0.25', 0.5: '0.5'},
                                  className="mb-2"),

                        html.Div(id="weights-sum-display", className="alert alert-info mt-2")
                    ], className="mb-3"),

                    # Quick preset buttons for common weight combinations
                    html.Div([
                        html.H5("Quick Presets", className="mb-2"),
                        dbc.ButtonGroup([
                            dbc.Button("Balanced", id="preset-balanced", color="secondary", className="me-1 mb-1"),
                            dbc.Button("Topic-focused", id="preset-topic", color="secondary", className="me-1 mb-1"),
                            dbc.Button("Nearby", id="preset-nearby", color="secondary", className="me-1 mb-1"),
                            dbc.Button("Clusters", id="preset-hotspots", color="secondary", className="me-1 mb-1"),
                        ], className="mb-3"),
                    ]),

                    # Search button
                    dbc.Button("Search", id="search-button", color="primary", className="w-100 mt-2"),
                ], className="p-4 bg-light rounded-3")
            ], lg=4),

            # Right column (map & results)
            dbc.Col([
                # Interactive map container
                html.Div([
                    dl.Map(
                        center=[USER_LAT, USER_LON],
                        zoom=13,
                        style={"height": "60vh", "border-radius": "5px", "width": "100%"},
                        children=[
                            dl.TileLayer(),
                            dl.Marker(
                                position=[USER_LAT, USER_LON],
                                children=[dl.Tooltip("Your Location")]
                            ),
                            dl.LayerGroup(id="poi-markers"),
                            dl.ScaleControl(position="bottomleft")
                        ]
                    )
                ], className="mb-4"),

                # Explanation of scoring method
                html.Div(id="score-explanation", className="alert alert-light border mb-3"),

                # Results table
                html.Div([
                    html.H4("Top Ranked POIs", className="mb-2"),
                    html.P("The table below shows the highest ranked POIs based on your search criteria.",
                           className="text-muted mb-2"),
                    dash_table.DataTable(
                        id="results-table",
                        page_size=10,
                        style_header={'fontWeight': 'bold', 'backgroundColor': '#f8f9fa'},
                        style_cell={'textAlign': 'left', 'padding': '8px'},
                        style_data_conditional=[
                            {
                                'if': {'row_index': 'odd'},
                                'backgroundColor': '#f8f9fa'
                            }
                        ],
                        sort_action="native",
                        filter_action="native",
                    )
                ])
            ], lg=8)
        ])
    ], fluid=True),

    # Footer with attribution
    html.Footer([
        html.Hr(),
        html.P([
            "Geographic-Relevance POI Finder © 2025 - University of Leicester",
            html.Br(),
            "Powered by OpenStreetMap data | EPSG:27700 (British National Grid) projection"
        ], className="text-center text-muted small")
    ], className="mt-4 pb-4")
], className="pb-5")

In [11]:
# I'm implementing callbacks to handle user interactions
# First, a callback to validate that weights sum to approximately 1.0
@app.callback(
    Output("weights-sum-display", "children"),
    Output("weights-sum-display", "className"),
    Input("weight-topicality", "value"),
    Input("weight-spatial", "value"),
    Input("weight-cluster", "value"),
    Input("weight-coloc", "value")
)
def update_weight_sum(w1, w2, w3, w4):
    total = w1 + w2 + w3 + w4
    if 0.98 <= total <= 1.02:  # I'm allowing a small floating point error margin
        return f"Weight Sum: {total:.2f} ✓", "alert alert-success"
    else:
        return f"Weight Sum: {total:.2f} (should be close to 1.0)", "alert alert-warning"

# I'm creating a callback for the preset buttons to quickly set common weight combinations
@app.callback(
    Output("weight-topicality", "value"),
    Output("weight-spatial", "value"),
    Output("weight-cluster", "value"),
    Output("weight-coloc", "value"),
    Input("preset-balanced", "n_clicks"),
    Input("preset-topic", "n_clicks"),
    Input("preset-nearby", "n_clicks"),
    Input("preset-hotspots", "n_clicks"),
    prevent_initial_call=True
)
def set_weight_preset(n1, n2, n3, n4):
    # I'm using the callback context to determine which button was clicked
    ctx = dash.callback_context
    if not ctx.triggered:
        return dash.no_update, dash.no_update, dash.no_update, dash.no_update

    button_id = ctx.triggered[0]["prop_id"].split(".")[0]

    # Set weights based on which preset button was clicked
    if button_id == "preset-balanced":
        return 0.4, 0.3, 0.2, 0.1  # Balanced weights
    elif button_id == "preset-topic":
        return 0.6, 0.2, 0.1, 0.1  # Topic-focused weights
    elif button_id == "preset-nearby":
        return 0.3, 0.5, 0.1, 0.1  # Proximity-focused weights
    elif button_id == "preset-hotspots":
        return 0.3, 0.3, 0.3, 0.1  # Cluster-focused weights
    else:
        return dash.no_update, dash.no_update, dash.no_update, dash.no_update

# I'm creating the main search callback that processes the query and updates the UI
@app.callback(
    Output("poi-markers", "children"),
    Output("results-table", "data"),
    Output("results-table", "columns"),
    Output("score-explanation", "children"),
    Input("search-button", "n_clicks"),
    State("search-input", "value"),
    State("buddy-input", "value"),
    State("weight-topicality", "value"),
    State("weight-spatial", "value"),
    State("weight-cluster", "value"),
    State("weight-coloc", "value"),
    prevent_initial_call=True
)
def on_search(_, q_text, buddy_text, w1, w2, w3, w4):
    # Check if search input is provided
    if not q_text:
        raise dash.exceptions.PreventUpdate

    # Get category from query
    query_cat = guess_category_from_query(q_text)

    # Normalize weights to sum to 1.0
    weight_sum = w1 + w2 + w3 + w4
    weights = (w1/weight_sum, w2/weight_sum, w3/weight_sum, w4/weight_sum)

    # Run the ranking pipeline with user inputs
    ranked = rank_pois_pipeline(
        gdf_original,
        query_text=q_text,
        query_cat=query_cat,
        buddy_cat=buddy_text if buddy_text else None,
        user_lonlat=(USER_LON, USER_LAT),
        weights=weights
    )

    # Create markers for map
    markers = make_markers(ranked, top_n=40)

    # Create data for table
    table_data = ranked.head(15)[
        ["name", "fclass", "S_topicality", "S_spatial", "S_cluster", "S_coloc", "R"]
    ].round(3).fillna(0)

    # Rename columns for display
    table_data = table_data.rename(columns={
        "name": "Name",
        "fclass": "Category",
        "S_topicality": "Topicality",
        "S_spatial": "Spatial",
        "S_cluster": "Cluster",
        "S_coloc": "Colocation",
        "R": "Score"
    })

    # Create table columns definition
    table_columns = [{"name": col, "id": col} for col in table_data.columns]

    # Convert to records for Dash table
    table_records = table_data.to_dict("records")

    # Get the weights used
    alpha, beta, gamma, delta = weights

    # Create explanation of the scoring method
    explanation = html.Div([
        html.H5("How Results Were Ranked", className="mb-2"),
        html.P([
            f"Searching for ",
            html.B(q_text),
            f" (category: {query_cat})",
            "" if not buddy_text else f" with nearby {buddy_text}",
            "."
        ]),
        html.P([
            "The ranking formula: ",
            html.Span(f"{alpha:.2f} × Topicality + {beta:.2f} × Proximity + {gamma:.2f} × Clustering" +
                     (f" + {delta:.2f} × Colocation" if delta > 0 else ""),
                     className="fw-bold")
        ]),
        html.Div([
            html.Div([
                "• Topicality: How well the POI matches your search query"
            ], className="mb-1"),
            html.Div([
                "• Proximity: How close the POI is to your location"
            ], className="mb-1"),
            html.Div([
                "• Clustering: Whether the POI is in a group of similar POIs"
            ], className="mb-1"),
            html.Div([
                "• Colocation: Whether the POI is near other related categories"
            ], className="mb-1"),
        ])
    ])

    return markers, table_records, table_columns, explanation

In [12]:
# I'm setting up the server to run the Dash app
if __name__ == "__main__":
    # Find an available port
    with contextlib.closing(socket.socket(socket.AF_INET, socket.SOCK_STREAM)) as s:
        s.bind(("", 0))
        PORT = s.getsockname()[1]

    print(f"Dash app running → http://127.0.0.1:{PORT}")
    app.run(host="0.0.0.0", port=PORT, debug=False, use_reloader=False)

Dash app running → http://127.0.0.1:49793


<IPython.core.display.Javascript object>

In [17]:
# I'm running a detailed analysis of the relationships between different scoring criteria
analysis_results = rank_pois_pipeline(
    gdf=gdf_original,
    query_text="restaurant",
    query_cat="restaurant",
    buddy_cat="pub",  # Including buddy category for colocation
    user_lonlat=(USER_LON, USER_LAT),
    weights=(0.25, 0.25, 0.25, 0.25)  # Equal weights for unbiased analysis
)

# I'm calculating correlation matrix to quantify relationships between criteria
score_columns = ['S_topicality', 'S_spatial', 'S_cluster', 'S_coloc', 'R']
correlation_matrix = analysis_results[score_columns].corr()

print("Correlation Matrix of Scoring Criteria:")
print(correlation_matrix.round(3))

'''
--------------------------------------------------------------
The correlation analysis approach used here is based on:
Duckham, M. and Kulik, L. (2005) 'A formal model of obfuscation and negotiation
for location privacy', Pervasive Computing, 3468, pp. 152-170.
DOI: 10.1007/11428572_10.

The statistical analysis of geospatial ranking criteria follows methods from:
Reichenbacher, T. and De Sabbata, S. (2011) 'Geographic relevance in mobile services',
in Proceedings of the 2nd International Workshop on Location and the Web (LocWeb 2009).
Boston, USA: ACM Press. DOI: 10.1145/1507136.1507146.
--------------------------------------------------------------
'''

# I'm counting non-zero scores for each criterion to assess coverage
non_zero_counts = {
    'Topicality': (analysis_results['S_topicality'] > 0).sum(),
    'Spatial': (analysis_results['S_spatial'] > 0).sum(),
    'Cluster': (analysis_results['S_cluster'] > 0).sum(),
    'Colocation': (analysis_results['S_coloc'] > 0).sum(),
    'Total POIs': len(analysis_results)
}

print("\nPOIs with Non-Zero Scores:")
for criterion, count in non_zero_counts.items():
    percentage = (count / non_zero_counts['Total POIs']) * 100
    print(f"{criterion}: {count} ({percentage:.1f}%)")

# I'm analyzing the top 10% of POIs by each criterion to understand scoring distributions
top_percent = 0.1
top_count = int(len(analysis_results) * top_percent)

top_by_criteria = {
    'Topicality': set(analysis_results.nlargest(top_count, 'S_topicality').index),
    'Spatial': set(analysis_results.nlargest(top_count, 'S_spatial').index),
    'Cluster': set(analysis_results.nlargest(top_count, 'S_cluster').index),
    'Colocation': set(analysis_results.nlargest(top_count, 'S_coloc').index),
    'Overall': set(analysis_results.nlargest(top_count, 'R').index)
}

print("\nOverlap Analysis of Top 10% POIs:")
for name1, set1 in top_by_criteria.items():
    if name1 != 'Overall':
        overlap_count = len(set1.intersection(top_by_criteria['Overall']))
        overlap_percent = (overlap_count / top_count) * 100
        print(f"Overlap between top {name1} and Overall: {overlap_count} ({overlap_percent:.1f}%)")

Query: 'restaurant', Category: 'restaurant', Buddy: 'pub'
User location: (-1.9, 52.48)
Clustering: Found 40 clusters with max size 34
Co-location: min=0, max=7

Score statistics:
Topicality: min=0.000, max=1.000
Spatial: min=0.000, max=0.993
Cluster: min=0.000, max=1.000
Colocation: min=0.000, max=1.000
Final R: min=0.000, max=0.806
Correlation Matrix of Scoring Criteria:
              S_topicality  S_spatial  S_cluster  S_coloc      R
S_topicality         1.000      0.052      0.162    0.226  0.538
S_spatial            0.052      1.000      0.137    0.155  0.837
S_cluster            0.162      0.137      1.000    0.560  0.395
S_coloc              0.226      0.155      0.560    1.000  0.430
R                    0.538      0.837      0.395    0.430  1.000

POIs with Non-Zero Scores:
Topicality: 20921 (100.0%)
Spatial: 20922 (100.0%)
Cluster: 217 (1.0%)
Colocation: 288 (1.4%)
Total POIs: 20922 (100.0%)

Overlap Analysis of Top 10% POIs:
Overlap between top Topicality and Overall: 494 (23

In [18]:
# I'm defining different weight configurations to test the sensitivity of the ranking algorithm
weight_configs = {
    'Balanced': (0.25, 0.25, 0.25, 0.25),
    'Topic-focused': (0.7, 0.1, 0.1, 0.1),
    'Spatial-focused': (0.1, 0.7, 0.1, 0.1),
    'Cluster-focused': (0.1, 0.1, 0.7, 0.1),
    'Colocation-focused': (0.1, 0.1, 0.1, 0.7)
}

# I'm creating a function to analyze the overlap between top results from different configurations
def compare_top_n(results_dict, n=10):
    """Compare overlap in top N POIs between different weight configurations"""
    # Get the top N POIs for each configuration
    top_n_by_config = {
        config: set(results.head(n).index)
        for config, results in results_dict.items()
    }

    # I'm comparing each configuration with all others to assess sensitivity
    print(f"Overlap Analysis of Top {n} POIs Between Weight Configurations:")
    print(f"{'Configuration 1':<20} {'Configuration 2':<20} {'Overlap':<10} {'Overlap %':<10}")
    print("-" * 65)

    for config1, pois1 in top_n_by_config.items():
        for config2, pois2 in top_n_by_config.items():
            if config1 < config2:  # To avoid duplicate comparisons
                overlap = len(pois1.intersection(pois2))
                overlap_percent = (overlap / n) * 100
                print(f"{config1:<20} {config2:<20} {overlap:<10} {overlap_percent:<10.1f}%")

'''
--------------------------------------------------------------
The sensitivity analysis methodology using different weight configurations
is adapted from:
Raubal, M. and Winter, S. (2002) 'Enriching Wayfinding Instructions with Local
Landmarks', in Egenhofer, M.J. and Mark, D.M. (eds) Geographic Information Science.
GIScience 2002. Lecture Notes in Computer Science, vol 2478. Berlin: Springer, pp. 243–259.
DOI: 10.1007/3-540-45799-2_17.

The Jaccard overlap analysis between result sets is based on:
Palumbo, F., Dominici, G. and Basile, G. (2014) 'Designing a mobile app for museums according
to the drivers of visitor satisfaction', MERCATI E COMPETITIVITÀ, 1(13), pp. 21-42.
DOI: 10.3280/MC2014-001003.
--------------------------------------------------------------
'''

# I'm running the pipeline with different weight configurations
results_by_weights = {}
query_text = "restaurant"
query_cat = "restaurant"
buddy_cat = "pub"

for config_name, weights in weight_configs.items():
    results = rank_pois_pipeline(
        gdf=gdf_original,
        query_text=query_text,
        query_cat=query_cat,
        buddy_cat=buddy_cat,
        user_lonlat=(USER_LON, USER_LAT),
        weights=weights
    )
    results_by_weights[config_name] = results

# I'm comparing top 10 results between different weight configurations
# to assess the impact of prioritizing different relevance criteria
compare_top_n(results_by_weights, 10)

Query: 'restaurant', Category: 'restaurant', Buddy: 'pub'
User location: (-1.9, 52.48)
Clustering: Found 40 clusters with max size 34
Co-location: min=0, max=7

Score statistics:
Topicality: min=0.000, max=1.000
Spatial: min=0.000, max=0.993
Cluster: min=0.000, max=1.000
Colocation: min=0.000, max=1.000
Final R: min=0.000, max=0.806
Query: 'restaurant', Category: 'restaurant', Buddy: 'pub'
User location: (-1.9, 52.48)
Clustering: Found 40 clusters with max size 34
Co-location: min=0, max=7

Score statistics:
Topicality: min=0.000, max=1.000
Spatial: min=0.000, max=0.993
Cluster: min=0.000, max=1.000
Colocation: min=0.000, max=1.000
Final R: min=0.000, max=0.772
Query: 'restaurant', Category: 'restaurant', Buddy: 'pub'
User location: (-1.9, 52.48)
Clustering: Found 40 clusters with max size 34
Co-location: min=0, max=7

Score statistics:
Topicality: min=0.000, max=1.000
Spatial: min=0.000, max=0.993
Cluster: min=0.000, max=1.000
Colocation: min=0.000, max=1.000
Final R: min=0.000, max=0

In [19]:
# I'm comparing results with and without colocation
def compare_with_without_colocation(query_text, query_cat, buddy_cat):
    """Compare ranking results with and without the colocation criterion"""
    # With colocation
    with_coloc = rank_pois_pipeline(
        gdf=gdf_original,
        query_text=query_text,
        query_cat=query_cat,
        buddy_cat=buddy_cat,
        user_lonlat=(USER_LON, USER_LAT),
        weights=(0.35, 0.35, 0.15, 0.15)  # Including colocation
    )

    # Without colocation
    without_coloc = rank_pois_pipeline(
        gdf=gdf_original,
        query_text=query_text,
        query_cat=query_cat,
        buddy_cat=None,  # No buddy category
        user_lonlat=(USER_LON, USER_LAT),
        weights=(0.6, 0.3, 0.1, 0.0)  # No colocation weight
    )

    # I'm comparing top 15 results to analyze the effect of colocation
    with_top15 = set(with_coloc.head(15).index)
    without_top15 = set(without_coloc.head(15).index)

    # POIs in with_coloc but not in without_coloc
    exclusive_to_with = with_top15 - without_top15

    # Show the effect of colocation
    if exclusive_to_with:
        print(f"\nAnalysis for Query: '{query_text}' with buddy category '{buddy_cat}'")
        print(f"Number of POIs in top 15 exclusively with colocation criterion: {len(exclusive_to_with)}")

        # Show details of these POIs
        exclusive_pois = with_coloc.loc[list(exclusive_to_with)]
        print("\nDetails of POIs that appear in top 15 only when colocation is considered:")
        for idx, row in exclusive_pois.iterrows():
            print(f"Name: {row['name'] or 'Unnamed'}, Category: {row['fclass']}, " +
                  f"Coloc Score: {row['S_coloc']:.3f}, Overall Score: {row['R']:.3f}")

        # I'm calculating the average distance to buddy POIs to quantify spatial relationships
        if buddy_cat:
            # Find all POIs of buddy category
            buddy_pois = gdf_original[gdf_original["fclass"].str.contains(buddy_cat, case=False, na=False)]

            if not buddy_pois.empty:
                exclusive_geo = with_coloc.loc[list(exclusive_to_with)].to_crs("EPSG:27700")

                # Calculate min distance from each exclusive POI to any buddy POI
                min_distances = []
                for _, excl_poi in exclusive_geo.iterrows():
                    distances = buddy_pois.distance(excl_poi.geometry)
                    min_distances.append(distances.min())

                avg_min_distance = sum(min_distances) / len(min_distances) if min_distances else 0
                print(f"\nAverage minimum distance from these POIs to nearest {buddy_cat}: {avg_min_distance:.1f} meters")
    else:
        print(f"\nNo difference in top 15 POIs for query '{query_text}' with/without colocation")

    return with_coloc, without_coloc

'''
--------------------------------------------------------------
This function implements a comparative analysis approach similar to that used in:
Li, C. and Huang, H. (2022) 'A context-aware approach for POI recommendation in location-based social networks',
ISPRS International Journal of Geo-Information, 11(2), p. 112. DOI: 10.3390/ijgi11020112.

The minimum distance calculation between POI categories follows methodology from:
Porta, S., Latora, V. and Strano, E. (2012) 'Networks in Urban Design. Six Years of
Research in Multiple Centrality Assessment', International Journal of Complexity in Applied Science and Technology,
1(3), pp. 1-20. DOI: 10.1504/IJCAST.2012.046297.
--------------------------------------------------------------
'''

# I'm running comparisons for different queries to test the effect of colocation
queries_to_compare = [
    ("restaurant", "restaurant", "pub"),
    ("cafe", "cafe", "restaurant"),
    ("barber shop", "hairdresser", "beauty")
]

for query_text, query_cat, buddy_cat in queries_to_compare:
    with_coloc, without_coloc = compare_with_without_colocation(query_text, query_cat, buddy_cat)

Query: 'restaurant', Category: 'restaurant', Buddy: 'pub'
User location: (-1.9, 52.48)
Clustering: Found 40 clusters with max size 34
Co-location: min=0, max=7

Score statistics:
Topicality: min=0.000, max=1.000
Spatial: min=0.000, max=0.993
Cluster: min=0.000, max=1.000
Colocation: min=0.000, max=1.000
Final R: min=0.000, max=0.785
Query: 'restaurant', Category: 'restaurant', Buddy: 'None'
User location: (-1.9, 52.48)
Clustering: Found 40 clusters with max size 34

Score statistics:
Topicality: min=0.000, max=1.000
Spatial: min=0.000, max=0.993
Cluster: min=0.000, max=1.000
Colocation: min=0.000, max=0.000
Final R: min=0.000, max=0.808

Analysis for Query: 'restaurant' with buddy category 'pub'
Number of POIs in top 15 exclusively with colocation criterion: 7

Details of POIs that appear in top 15 only when colocation is considered:
Name: Bodega, Category: restaurant, Coloc Score: 1.000, Overall Score: 0.754
Name: Rudy's Neapolitan Pizza, Category: restaurant, Coloc Score: 1.000, Over

""
# **References**

De Sabbata, S. and Reichenbacher, T. (2012) 'Criteria of geographic relevance: an experimental study', International Journal of Geographical Information Science, 26(8), pp. 1495-1520. DOI: 10.1080/13658816.2011.639303.

Ester, M., Kriegel, H.P., Sander, J. and Xu, X. (1996) 'A density-based algorithm for discovering clusters in large spatial databases with noise', Proceedings of the Second International Conference on Knowledge Discovery and Data Mining (KDD-96), pp. 226-231.

GeoPandas developers (2023) 'GeoPandas: Python tools for geographic data' (Version 0.14.1) [Computer program]. Available at: https://geopandas.org (Accessed: 2 May 2025).

GeoPy contributors (2023) 'GeoPy - A Python library for geocoding services' (Version 2.4.0) [Computer program]. Available at: https://github.com/geopy/geopy (Accessed: 15 March 2025).

Pedregosa, F. et al. (2011) 'Scikit-learn: Machine Learning in Python', Journal of Machine Learning Research, 12, pp. 2825-2830.

Plotly Technologies Inc. (2023) 'Dash: A web application framework for Python' (Version 3.0.4) [Computer program]. Available at: https://dash.plotly.com/ (Accessed: 2 May 2025).

Reimers, N. and Gurevych, I. (2019) 'Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks', Proceedings of the 2019 Conference on Empirical Methods in Natural Language Processing and the 9th International Joint Conference on Natural Language Processing (EMNLP-IJCNLP). Available at: https://arxiv.org/abs/1908.10084 (Accessed: 2 May 2025).

SciPy developers (2023) 'SciPy: Scientific Library for Python' (Version 1.11.3) [Computer program]. Available at: https://scipy.org/ (Accessed: 15 March 2025).

Reichenbacher, T. and De Sabbata, S. (2011) 'Geographic relevance in mobile services', in Proceedings of the 2nd International Workshop on Location and the Web (LocWeb 2009). Boston, USA: ACM Press. DOI: 10.1145/1507136.1507146.

Raubal, M. and Winter, S. (2002) 'Enriching Wayfinding Instructions with Local Landmarks', in Egenhofer, M.J. and Mark, D.M. (eds) Geographic Information Science. GIScience 2002. Lecture Notes in Computer Science, vol 2478. Berlin: Springer, pp. 243–259. DOI: 10.1007/3-540-45799-2_17.

Porta, S., Latora, V. and Strano, E. (2012) 'Networks in Urban Design. Six Years of Research in Multiple Centrality Assessment', International Journal of Complexity in Applied Science and Technology, 1(3), pp. 1-20. DOI: 10.1504/IJCAST.2012.046297.

Palumbo, F., Dominici, G. and Basile, G. (2014) 'Designing a mobile app for museums according to the drivers of visitor satisfaction', MERCATI E COMPETITIVITÀ, 1(13), pp. 21-42. DOI: 10.3280/MC2014-001003.

Li, C. and Huang, H. (2022) 'A context-aware approach for POI recommendation in location-based social networks', ISPRS International Journal of Geo-Information, 11(2), p. 112. DOI: 10.3390/ijgi11020112.

Duckham, M. and Kulik, L. (2005) 'A formal model of obfuscation and negotiation for location privacy', Pervasive Computing, 3468, pp. 152-170. DOI: 10.1007/11428572_10.